# DS2002 · Pandas Core Ops

**Studio — 2026-09-16 · Fall 2026**  
**Class time:** 45 minutes

---

## Two halves today

**Part 1** is a short drill on the copy trap from Monday, because it is the bug that quietly changes your numbers instead of raising an error. Budget about 15 minutes.

**Part 2** is one deliverable end to end: a per-vendor summary a game-day manager could act on. Three sources, a join that does not behave, and a report at the end.

The Part 2 data has problems planted in it. Finding them is part of the work — a report you cannot defend is worth nothing, however good the code looks.

---

## Part 1 — The copy trap, with the damage visible

Monday you saw `.copy()` on a slide. Here you watch what happens without it, which is the only way it sticks.

Run the next cell for a small frame to experiment on. The real files arrive in Part 2.

In [ ]:
import pandas as pd

print('pandas', pd.__version__)

sales = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'item': ['Rain Poncho', 'Cheeseburger', 'Rain Poncho', 'Hot Dog', 'Foam Finger'],
    'qty': [5, 2, 8, 3, 1],
    'price': [6.00, 7.50, 6.00, 4.50, 12.00],
})
sales

### Bad result 1 — the edit that goes nowhere

The ponchos went on sale at $4.50. This is how nearly everybody writes it the first time. Run it, and read the two printed prices before you read any warning.

In [ ]:
print('before:', sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

sales[sales['item'] == 'Rain Poncho']['price'] = 4.50

print('after: ', sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

Nothing changed, and nothing crashed.

That line is two operations, not one. `sales[sales['item'] == 'Rain Poncho']` builds a brand-new temporary frame holding the two poncho rows. `['price'] = 4.50` then sets the price **on that temporary**. Nothing was holding a reference to it, so Python discarded it a microsecond later. Your data never saw the change.

This is the expensive kind of bug. The code looks right, it does not raise, and the number you report is the old one.

### Bad result 2 — the edit that lands somewhere you did not mean

Now the version where the slice goes into a variable first.

In [ ]:
rain = sales[sales['item'] == 'Rain Poncho']
rain['sale_price'] = 4.50

print(rain)
print()
print("'sale_price' in rain? ", 'sale_price' in rain.columns)
print("'sale_price' in sales?", 'sale_price' in sales.columns)

This one did something — `rain` has the new column. What it did not do is touch `sales`.

Whether you also get a warning here depends on your pandas version, which is exactly why the printed columns matter more than the warnings:

- On **pandas 2.x** you get `SettingWithCopyWarning`, because older pandas could not promise whether `rain` shared memory with `sales`.
- On **pandas 3.x** you get nothing at all. Copy-on-Write is always on, so `rain` is guaranteed to be its own frame.

The warning was never the lesson. The lesson is that you have to know which frame your assignment lands in, and slicing never hands you the original.

### The fix — two tools, for two different jobs

Decide what you want before you type. There are exactly two answers.

In [ ]:
# Tool 1 -- you want a separate frame to work on. Say so with .copy().
rain = sales[sales['item'] == 'Rain Poncho'].copy()
rain['sale_price'] = 4.50
print('rain has sale_price   :', 'sale_price' in rain.columns)
print('sales left alone      :', 'sale_price' not in sales.columns)
print()

# Tool 2 -- you want to change the original. One operation, so it lands.
sales.loc[sales['item'] == 'Rain Poncho', 'price'] = 4.50
print('sales poncho price now:',
      sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

Keep this table until it is reflex:

| What you want | What to write |
|---|---|
| A separate frame you can modify freely | `sub = df[mask].copy()` |
| To change the original in place | `df.loc[mask, 'col'] = value` |
| Nothing, ever | `df[mask]['col'] = value` |

The third row is not a style preference. It does not work.

### Your turn 1 — make the change actually land

The Hot Dog price should be `5.00` in `sales` itself. The broken version is sitting in the cell as a comment; do not use it. Write the version that works, then print the price to prove it.

In [ ]:
# Broken -- leave it commented, it silently does nothing:
#     sales[sales['item'] == 'Hot Dog']['price'] = 5.00

# TODO: the version that changes `sales`

# TODO: print the Hot Dog price -- expect [5.0]

### Your turn 2 — a working copy that leaves the original alone

Build `bulk`: only the rows with `qty >= 3`, plus a new `line_total` column equal to `qty * price`. `sales` must come out of this unchanged.

Two things must be true when you are done: `bulk` has a `line_total` column, and `sales` does not.

In [ ]:
# TODO: bulk = ...
# TODO: bulk['line_total'] = ...

# Uncomment these to check yourself:
# assert 'line_total' in bulk.columns
# assert 'line_total' not in sales.columns
# print(bulk)

---

## Part 2 — The vendor report

Three sources: order lines, a vendor roster, and a revenue target per zone. Run the next cell to load them.

In [ ]:
import pandas as pd
from io import StringIO

orders = pd.read_csv(StringIO('''order_id,vendor_id,item,qty,price
1,V-01,Cheeseburger,2,7.50
2,V-10,Foam Finger,1,12.00
3,V-01,Hot Dog,3,4.50
4,V-18,Rain Poncho,5,6.00
5,V-10,UVA T-Shirt,1,24.00
6,V-05,Chicken Tacos,4,6.50
7,V-18,Rain Poncho,8,6.00
8,V-42,Kettle Corn,3,5.00'''))

vendors = pd.read_csv(StringIO('''vendor_id,vendor_name,zone
V-01,Hoos Burgers,A
V-05,Rotunda Tacos,B
V-10,Cav Merch North,A
V-18,Rally Rain Gear,C
V-18,Rally Rain Gear,C'''))

targets = pd.read_csv(StringIO('''zone,revenue_target
A,80
B,25
C,60'''))

print('orders:', orders.shape, '| vendors:', vendors.shape, '| targets:', targets.shape)
orders

### Worked example — the merge, done carefully

Here is one merge done properly, so the pattern is on the screen before you write anything. Three things happen: record the baseline, merge with `indicator=True`, then compare against the baseline.

In [ ]:
baseline_rows = len(orders)
orders['revenue'] = orders['qty'] * orders['price']
baseline_revenue = orders['revenue'].sum()
print(f'before: {baseline_rows} rows, ${baseline_revenue:.2f}')

check = orders.merge(vendors, on='vendor_id', how='left', indicator=True)
print(f'after:  {len(check)} rows, ${check["revenue"].sum():.2f}')
print()
print(check['_merge'].value_counts())

Eight orders went in and **ten** came out, and the revenue total jumped by $78. Both symptoms point at the same cause, and it is in the `vendors` table, not in the orders.

Note that `_merge` says `both = 9` and `left_only = 1`. Nine matches out of eight orders is already impossible, which is the tell.

Find it before you go further — everything downstream inherits this bug.

In [ ]:
# Which vendor_id appears more than once in the vendor list?
print(vendors['vendor_id'].value_counts())
print()
print('duplicated vendor rows:', vendors.duplicated().sum())

### Build 1 — fix the vendor list, then merge

**TODO:** drop the duplicate vendor row, then join it onto `orders` with an indicator. Your merge must come out at **8 rows** with the revenue total unchanged from the baseline. Print both to prove it.

In [ ]:
# TODO: clean_vendors = ...
# TODO: joined = orders.merge(...)
# TODO: print row count and revenue, and compare to baseline_rows / baseline_revenue

### Build 2 — handle the vendor nobody has heard of

One order belongs to a vendor that is not on the roster. You have three options, and this is a judgment call:

1. Drop it — clean report, understated revenue.
2. Keep it with a blank name — the revenue total stays right, but it shows up as `NaN` in every chart and table.
3. Label it `'Unknown vendor'` and keep it in a zone called `'Unassigned'`.

**TODO:** pick one, implement it, and write one sentence saying why. Print how much revenue the decision affects either way.

In [ ]:
# TODO: implement your choice
# TODO: print the revenue attached to the unmatched order

**My decision, and why:** _..._

### Build 3 — the per-vendor summary

**TODO:** one row per vendor, with:

- `orders` — how many orders
- `units` — total quantity
- `revenue` — total revenue
- `avg_ticket` — average revenue per order, rounded to 2 decimals

Sorted by revenue, highest first. Use `.agg()` with named outputs so the columns come out with the names above.

In [ ]:
# TODO

### Build 4 — did each zone hit its target?

**TODO:** total revenue by zone, join `targets` on, and add a `hit_target` boolean column. Then print a one-line sentence for each zone that a manager could read.

In [ ]:
# TODO

### Build 5 — the one number that matters *(stretch, if you have time)*

**TODO:** rain gear is the thing we can actually act on. Print total poncho units sold and what share of overall revenue they represent, as a percentage rounded to one decimal.

In [ ]:
# TODO

---

## Checkpoint (participation)

Report the one line that fixes the copy trap, your row count and revenue total after the merge, and what you decided to do with the unknown vendor.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [ ]:
# Checkpoint
copy_fix = 'TODO'          # TODO: the line you wrote to change `sales` itself
rows_after_merge = None    # TODO: should equal 8
revenue_after_merge = None # TODO: should equal the baseline
unknown_vendor_call = 'TODO'  # TODO: what you did with V-42, and why
top_vendor = 'TODO'        # TODO: highest-revenue vendor from your Build 3 summary

print('copy fix:', copy_fix)
print('rows after merge:', rows_after_merge)
print('revenue after merge:', revenue_after_merge)
print('unknown vendor:', unknown_vendor_call)
print('top vendor:', top_vendor)